In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from scipy import stats

In [0]:
olist_orders = spark.read.table('olist_ecommerce.default.olist_order_records')
olist_orders.show(5)

In [0]:

orders = olist_orders.withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp")
        ))

first_orders = orders.filter(F.col("rn") == 1)
second_orders = orders.filter(F.col("rn") == 2).withColumnRenamed('order_purchase_timestamp',
                                                                  'next_purchase_timestamp')

orders = first_orders.alias('fo').join(second_orders.alias('so'),
                                        F.col('fo.customer_unique_id') == F.col('so.customer_unique_id'),
                                        'inner') \
                    .select('fo.*', 'so.next_purchase_timestamp').drop('rn')
orders.show()


### Hypothesis test 1

* H₀: Monetary spend does not significantly influence the time gap until the next purchase.
* H₁: Monetary spend significantly influences the time gap until the next purchase.


In [0]:
hypo_1_df = orders.select("customer_unique_id", 
                          F.date_diff('next_purchase_timestamp', 'order_purchase_timestamp')
                          .alias('days_gap'), 'total_cost')
quant = hypo_1_df.approxQuantile("total_cost", [0.25, 0.5, 0.75], 0.05)
hypo_1_df = hypo_1_df.withColumn(
    "spenders",
    F.when(F.col("total_cost") < quant[0], F.lit('penny'))
     .when(F.col("total_cost") < quant[1], F.lit('low'))
     .when(F.col("total_cost") < quant[2], F.lit('medium'))
     .otherwise(F.lit('high'))
)
hypo_1_df.show(5)
hypo_1_pdf = hypo_1_df.select("days_gap", "spenders").toPandas()
penny = hypo_1_pdf[hypo_1_pdf['spenders'] == 'penny']['days_gap']
low = hypo_1_pdf[hypo_1_pdf['spenders'] == 'low']['days_gap']
medium = hypo_1_pdf[hypo_1_pdf['spenders'] == 'medium']['days_gap']
high = hypo_1_pdf[hypo_1_pdf['spenders'] == 'high']['days_gap']

# anova test
f_stat, p_value = stats.f_oneway(penny, low, medium, high)
print(f'f_stat : {f_stat}')
print(f'p_value : {p_value}')